# §13.11.5 — 예측을 고정한 채 분포를 바꾸는 최적화의 재현

> 딥러닝 교재 · 3부 13장 11절 5항 (🐍)
> 선행: §13.11.1(반증 1) · §13.11.2(반증 2) · §13.11.3(값 크기) · §13.11.4(유용 조건)

## 이 노트북이 답하는 질문

1. **같은 예측을 내는 전혀 다른 어텐션 분포가 존재하는가?** 직접 최적화로 찾아본다.
2. **어텐션과 기울기 귀속의 상관은 깊이에 어떻게 의존하는가?** §13.11.4의 조건을 실측한다.
3. **가중치가 놓치는 것은 크기만이 아니다.** 부호 있는 기여와 대조한다.

**예상 실행 시간** CPU 약 60초 (`FAST = True`이면 약 30초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 두 모델 — 얕은 풀링과 혼합 한 층을 얹은 버전

과제는 §13.2.4의 표지 분류를 변형해 **같은 부류의 표지가 두 개** 있게 한다 —
정답에는 둘 중 하나면 충분하므로 "옳은 어텐션"이 유일하지 않다(반증 1이 겨냥하는
여유가 구조적으로 존재한다).

모델은 둘을 준비한다. **얕은 모델**은 어텐션 풀링 → 판독. **깊은 모델**은 그 앞에
자기 어텐션 혼합 한 층을 얹는다. 관찰 대상인 풀링 어텐션 $\alpha$는 두 모델에서
같은 자리에 있지만, 깊은 쪽에서는 $\alpha$가 보는 표현이 이미 토큰을 섞은 뒤다.

In [ ]:
T_SEQ = 16
N_CLS = 8
D = 48
rn0 = np.random.default_rng(SEED % 99991)
E_CLS = rn0.standard_normal((N_CLS, D))
E_MARK = rn0.standard_normal(D)

def make_batch(B, rn):
    cls = rn.integers(0, N_CLS, (B, T_SEQ))
    X = E_CLS[cls].copy()
    y = rn.integers(0, N_CLS, B)
    for b in range(B):
        p1, p2 = rn.permutation(T_SEQ)[:2]
        for pp in (p1, p2):
            X[b, pp] = E_CLS[y[b]] + E_MARK
    return X, y

def init_model(seed, deep):
    rn = np.random.default_rng(seed)
    m = {'Wk': rn.standard_normal((D, D)) / np.sqrt(D),
         'Wv': rn.standard_normal((D, D)) / np.sqrt(D),
         'u': rn.standard_normal(D),
         'W': rn.standard_normal((D, N_CLS)) / np.sqrt(D)}
    if deep:
        for nm in ['Mq', 'Mk', 'Mv']:
            m[nm] = rn.standard_normal((D, D)) / np.sqrt(D)
    return m

def fwd(m, X, deep, want_attr=False, y=None):
    B = X.shape[0]; aux = {}
    if deep:                                            # 혼합 층 (잔차)
        Qm, Km, Vm = X @ m['Mq'], X @ m['Mk'], X @ m['Mv']
        em = np.einsum('btd,bsd->bts', Qm, Km) / np.sqrt(D)
        em -= em.max(2, keepdims=True)
        am = np.exp(em); am /= am.sum(2, keepdims=True)
        H = X + np.einsum('bts,bsd->btd', am, Vm)
        aux.update(am=am, Qm=Qm, Km=Km, Vm=Vm)
    else:
        H = X
    K = H @ m['Wk']; Vv = H @ m['Wv']
    e = (K @ m['u']) / np.sqrt(D); e -= e.max(1, keepdims=True)
    a = np.exp(e); a /= a.sum(1, keepdims=True)
    pooled = np.einsum('bt,btd->bd', a, Vv)
    logits = pooled @ m['W']
    z = logits - logits.max(1, keepdims=True)
    P = np.exp(z); P /= P.sum(1, keepdims=True)
    out = dict(a=a, P=P, V=Vv, K=K, H=H, aux=aux)
    if want_attr:                                       # 정답 로짓의 입력 기울기 귀속
        dlogit = np.zeros_like(P); dlogit[np.arange(B), y] = 1.0
        dpool = dlogit @ m['W'].T
        da = np.einsum('bd,btd->bt', dpool, Vv)
        dV = a[:, :, None] * dpool[:, None, :]
        de = a * (da - (a * da).sum(1, keepdims=True)) / np.sqrt(D)
        dK = de[:, :, None] * m['u'][None, None, :]
        dH = dK @ m['Wk'].T + dV @ m['Wv'].T
        if deep:
            am, Qm, Km, Vm = aux['am'], aux['Qm'], aux['Km'], aux['Vm']
            dVm = np.einsum('bts,btd->bsd', am, dH)
            dam = np.einsum('btd,bsd->bts', dH, Vm)
            dem = am * (dam - (am * dam).sum(2, keepdims=True)) / np.sqrt(D)
            dQm = np.einsum('bts,bsd->btd', dem, Km)
            dKm = np.einsum('bts,btd->bsd', dem, Qm)
            dX = dH + dQm @ m['Mq'].T + dKm @ m['Mk'].T + dVm @ m['Mv'].T
        else:
            dX = dH
        out['attr'] = np.linalg.norm(dX, axis=2)
    return out

def train(deep, seed=0, steps=None, lr=2e-3):
    steps = steps or (200 if FAST else 400)
    m = init_model(seed, deep)
    ms = {k: np.zeros_like(v) for k, v in m.items()}
    vs = {k: np.zeros_like(v) for k, v in m.items()}
    rb = np.random.default_rng(900 + seed)
    for t in range(1, steps + 1):
        X, y = make_batch(64, rb)
        o = fwd(m, X, deep)
        B = 64
        dlog = o['P'].copy(); dlog[np.arange(B), y] -= 1; dlog /= B
        H, a, Vv, K = o['H'], o['a'], o['V'], o['K']
        pooled = np.einsum('bt,btd->bd', a, Vv)
        g = {'W': pooled.T @ dlog}
        dpool = dlog @ m['W'].T
        da = np.einsum('bd,btd->bt', dpool, Vv)
        dV = a[:, :, None] * dpool[:, None, :]
        de = a * (da - (a * da).sum(1, keepdims=True)) / np.sqrt(D)
        g['u'] = np.einsum('bt,btd->d', de, K)
        dK = de[:, :, None] * m['u'][None, None, :]
        g['Wk'] = np.einsum('btd,bte->de', H, dK)
        g['Wv'] = np.einsum('btd,bte->de', H, dV)
        dH = dK @ m['Wk'].T + dV @ m['Wv'].T
        if deep:
            am, Qm, Km, Vm = o['aux']['am'], o['aux']['Qm'], o['aux']['Km'], o['aux']['Vm']
            dVm = np.einsum('bts,btd->bsd', am, dH)
            dam = np.einsum('btd,bsd->bts', dH, Vm)
            dem = am * (dam - (am * dam).sum(2, keepdims=True)) / np.sqrt(D)
            dQm = np.einsum('bts,bsd->btd', dem, Km)
            dKm = np.einsum('bts,btd->bsd', dem, Qm)
            g['Mq'] = np.einsum('btd,bte->de', X, dQm)
            g['Mk'] = np.einsum('btd,bte->de', X, dKm)
            g['Mv'] = np.einsum('btd,bte->de', X, dVm)
        for k in g:
            ms[k] = 0.9 * ms[k] + 0.1 * g[k]
            vs[k] = 0.999 * vs[k] + 0.001 * g[k] ** 2
            m[k] -= lr * (ms[k] / (1 - 0.9 ** t)) / (np.sqrt(vs[k] / (1 - 0.999 ** t)) + 1e-8)
    return m

X_ev, y_ev = make_batch(256, np.random.default_rng(123))
m_sh = train(False, seed=1)
m_dp = train(True, seed=1)
o_sh = fwd(m_sh, X_ev, False, want_attr=True, y=y_ev)
o_dp = fwd(m_dp, X_ev, True, want_attr=True, y=y_ev)
print(f"정확도: 얕은 {(o_sh['P'].argmax(1)==y_ev).mean():.3f} | 깊은 {(o_dp['P'].argmax(1)==y_ev).mean():.3f}")

---
## 2. 반증 1 — 예측을 고정하고 분포를 밀어낸다

예제마다 풀링 로짓 $u_e$를 자유 변수로 두고, 출력 분포를 지키는 벌점 아래 원본
$\alpha$에서 최대한 멀어지게 최적화한다(잡음 초기화로 대칭을 깬다):
$\min_{u_e}\ -\|\alpha_{u_e}-\alpha\|^2 + \lambda\,\mathrm{KL}(P_{u_e}\|P)$.

In [ ]:
def adv_attention(m, X, a0, P0, lam=30.0, steps=200, lr=0.2):
    Vv = X @ m['Wv']
    rn = np.random.default_rng(7)
    ue = np.log(a0 + 1e-9) + 1.5 * rn.standard_normal(a0.shape)
    mom = np.zeros_like(ue)
    for it in range(steps):
        ue_ = ue - ue.max(1, keepdims=True)
        a = np.exp(ue_); a /= a.sum(1, keepdims=True)
        pooled = np.einsum('bt,btd->bd', a, Vv)
        logits = pooled @ m['W']
        z = logits - logits.max(1, keepdims=True)
        P = np.exp(z); P /= P.sum(1, keepdims=True)
        dl_da = -2 * (a - a0) + np.einsum('bd,btd->bt', (P - P0) @ m['W'].T * lam, Vv)
        due = a * (dl_da - (a * dl_da).sum(1, keepdims=True))
        mom = 0.9 * mom + due
        ue -= lr * mom
    ue_ = ue - ue.max(1, keepdims=True)
    a = np.exp(ue_); a /= a.sum(1, keepdims=True)
    pooled = np.einsum('bt,btd->bd', a, Vv)
    logits = pooled @ m['W']
    z = logits - logits.max(1, keepdims=True)
    P = np.exp(z); P /= P.sum(1, keepdims=True)
    return a, P

a_adv, P_adv = adv_attention(m_sh, X_ev, o_sh['a'], o_sh['P'])
tv = 0.5 * np.abs(a_adv - o_sh['a']).sum(1)
dpred = 0.5 * np.abs(P_adv - o_sh['P']).sum(1)
same_pred = (P_adv.argmax(1) == o_sh['P'].argmax(1)).mean()
print(f"어텐션 이동 TV 중앙값 {np.median(tv):.2f} | 출력 이동 TV 중앙값 {np.median(dpred):.3f} | 예측 유지 {same_pred:.2f}")

---
## 3. 반증 2의 조건성, 그리고 부호를 모르는 가중치

In [ ]:
from scipy.stats import spearmanr
rho_sh = np.array([spearmanr(o_sh['a'][b], o_sh['attr'][b]).statistic for b in range(256)])
rho_dp = np.array([spearmanr(o_dp['a'][b], o_dp['attr'][b]).statistic for b in range(256)])
print(f"Spearman(α, 기울기 귀속): 얕은 모델 중앙값 {np.median(rho_sh):.2f} | 깊은 모델 {np.median(rho_dp):.2f}")

# 부호 있는 기여: c_j = α_j (v_j · w_y) — 정답 로짓에의 실제 기여
w_y = m_dp['W'][:, y_ev].T                              # (B,D)
signed = o_dp['a'] * np.einsum('btd,bd->bt', o_dp['V'], w_y)
neg_frac = (signed < 0).mean()
print(f"깊은 모델에서 로짓 기여가 음수인 토큰 비율: {neg_frac:.2f} — 가중치 α는 부호를 모른다")

---
## 4. 교재 그림 — fig_13_11_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 한 예제: 원본 대 적대 분포
ax = axes[0]
b0 = int(np.argmax(tv * (dpred < 0.01)))
xs = np.arange(T_SEQ); w = 0.4
ax.bar(xs - w/2, o_sh['a'][b0], w, color=CB[5], label=lab('원본 $\\alpha$', 'original'))
ax.bar(xs + w/2, a_adv[b0], w, color=CB[4], label=lab('적대 $\\alpha$ (같은 예측)', 'adversarial'))
ax.set_xlabel(lab('토큰 위치', 'position'))
ax.set_ylabel(lab('어텐션 가중치', 'attention'))
ax.set_title(lab(f'(a) 다른 분포, 같은 예측 (TV={tv[b0]:.2f}, Δ출력={dpred[b0]:.3f})',
                 '(a) same prediction'), fontsize=10)
ax.legend(fontsize=8)

# (b) 사례 전체
ax = axes[1]
ax.scatter(tv, dpred, s=9, alpha=0.5, color=CB[5])
ax.set_xlabel(lab('어텐션 분포의 이동 (TV)', 'attention shift'))
ax.set_ylabel(lab('출력 분포의 이동 (TV)', 'output shift'))
ax.set_title(lab('(b) 분포는 절반이 움직여도 예측은 그대로', '(b) achievable shifts'), fontsize=10)

# (c) 귀속 상관 — 깊이의 효과
ax = axes[2]
bins = np.linspace(-0.6, 1.0, 33)
ax.hist(rho_sh, bins=bins, alpha=0.7, color=CB[5], label=lab('얕은 모델 (풀링만)', 'shallow'))
ax.hist(rho_dp, bins=bins, alpha=0.7, color=CB[4], label=lab('깊은 모델 (혼합 1층)', 'deep'))
ax.set_xlabel(lab('스피어만 상관 $\\rho(\\alpha,$ 기울기 귀속$)$', 'Spearman corr'))
ax.set_ylabel(lab('예제 수', 'count'))
ax.set_title(lab('(c) 층 하나가 끼면 상관이 무너진다', '(c) depth breaks the link'), fontsize=10)
ax.legend(fontsize=8)

# (d) 부호 있는 기여
ax = axes[3]
ax.scatter(o_dp['a'][::3].ravel(), signed[::3].ravel(), s=5, alpha=0.25, color=CB[5])
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel(lab('가중치 $\\alpha_j$', 'weight'))
ax.set_ylabel(lab('정답 로짓 기여 $\\alpha_j\\,(v_j\\cdot w_y)$', 'signed contribution'))
ax.set_title(lab(f'(d) 같은 가중치, 반대 부호의 기여 (음수 {neg_frac*100:.0f}%)', '(d) sign-blind weights'), fontsize=10)

save_book_fig(fig, 'fig_13_11_5')
plt.show()

> ### 읽는 법
>
> (a)–(b) 반증 1의 재현. 출력을 사실상 바꾸지 않으면서 어텐션 질량의 절반을 옮기는
> 대안 분포가 대부분의 예제에서 발견된다(TV 중앙값 0.5, 예측 유지 100%). 같은 예측에
> "모델이 본 곳"이 여러 개라면 그중 하나를 **그** 설명으로 제시할 근거가 없다(§13.11.1).
> (c) 반증 2를 조건과 함께 재현한 것이 이 그림의 수확이다. 풀링뿐인 얕은 모델에서는
> 어텐션과 기울기 귀속이 거의 일치하지만(중앙값 0.99 — §13.11.4의 "얕으면 제한적으로
> 유용"의 실측), **혼합 층 하나만 끼어도** 상관이 무너진다(중앙값 0.3대). 깊은 모델의
> $\alpha$는 이미 섞인 표현들 사이의 가중치라서, "입력 토큰의 중요도"와의 연결이
> 층을 지날수록 흐려지는 것이다.
> (d) 가장 단순한 반박의 강화판. 가중치는 크기 $\|v\|$만이 아니라 **부호**도 모른다 —
> 같은 $\alpha$라도 정답 로짓을 올리는 토큰과 내리는 토큰이 섞여 있다(§13.11.3).

---
## 5. 자기 점검

1. (a)의 적대 분포는 질량을 어디로 옮겼는가? 같은 부류 표지가 둘이라는 과제 설계와 연결하라.
2. 표지를 하나만 두면 (b)의 달성 가능 TV가 얼마나 줄어드는가? 실행해 보고, "여유"의 출처를 확인하라.
3. (c)에서 혼합 층을 두 개로 늘리면 상관이 더 내려가는가? 층수를 훑어 §13.11.4의 조건을 곡선으로 그려 보라.
4. (c)의 낮은 상관이 "기울기 귀속이 옳다"를 뜻하지 않는 이유는? §46장의 귀속 방법 비판을 예습하는 질문이다.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `lam` | 2절 | 30 | 예측 고정의 엄격함 |
| 표지 수 | 1절 | 2 | 대안 분포의 여유 |
| 혼합 층 수 | 1절 | 0/1 | (c)의 조건 곡선 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")